# Parametric Study - energy sweep

Run the same He -> Fe simulation at several beam energies and compare the
total vacancies produced per ion.  The pattern generalizes to any parameter
sweep (material, fluence, displacement threshold, ...).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import opentrim

In [ ]:
def make_config(energy_eV, n_ions=3000):
    cfg = opentrim.Config()
    cfg.IonBeam.ion = opentrim.Element('He')
    cfg.IonBeam.energy_distribution.center = energy_eV

    mat = opentrim.Material(); mat.id = 'Fe'; mat.density = 7.874
    atom = opentrim.Atom(); atom.element = opentrim.Element('Fe')
    atom.X = 1.0; atom.Ed = 40.0; atom.El = 3.0; atom.Es = 4.28; atom.Er = 40.0; atom.Rc = 0.946
    mat.composition.append(atom)
    cfg.Target.materials.append(mat)

    region = opentrim.Region(); region.id = 'slab'; region.material_id = 'Fe'
    region.origin = [0, 0, 0]; region.size = [200, 100, 100]
    cfg.Target.regions.append(region)

    cfg.Target.cell_count = [400, 1, 1]
    cfg.Target.size = [200.0, 100.0, 100.0]
    cfg.Simulation.electronic_stopping = opentrim.Stopping.SRIM13
    cfg.Simulation.simulation_type = opentrim.SimulationType.FullCascade
    # output base name allows only [0-9a-zA-Z_], so build it from integer keV
    cfg.Output.outfilename = f'He_{int(round(energy_eV / 1e3))}keV_Fe'
    cfg.Run.max_no_ions = n_ions
    cfg.Run.threads = 4
    cfg.Run.seed = 1
    cfg.validate()
    return cfg

In [ ]:
energies = [1e6, 2e6, 3e6, 5e6]   # eV
total_vac = []

for E in energies:
    sim = opentrim.Driver()
    sim.init(make_config(E))
    sim.run()
    sim.wait()
    info = opentrim.Info(sim)
    v, _ = info['tally']['damage_events']['Vacancies']
    total_vac.append(float(v.sum()))   # vacancies per ion, summed over depth
    print(f'{E/1e6:.0f} MeV -> {total_vac[-1]:.2f} vacancies/ion')

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(np.array(energies) / 1e6, total_vac, 'o-')
plt.xlabel('Beam energy (MeV)')
plt.ylabel('Total vacancies per ion')
plt.title('He -> Fe: damage vs energy')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()